# Experiment 1: Comparing Loss Functions on Real Face Data (FairFace + UTKFace)

We hold **every other layer constant** (same CNN, ReLU, BatchNorm, AdamW, LR, seed) and swap **only the Loss Function**:

1. **FairFace (Classification — Origin, Gender, Age Groups):**
   * `cross_entropy` — Standard `-log(p_true)`
   * `label_smoothing` — Soft targets (`0.1` smoothing) to reduce overconfidence
   * `focal` — `-(1 - p_t)^2 * log(p_t)` focuses on hard/minority examples
   * `mse_onehot` — MSE on softmax probabilities (watch gradient saturation!)
2. **UTKFace (Continuous Age Regression + Imbalanced Origin):**
   * `mse` (L2) — Penalizes large age errors quadratically
   * `mae` (L1) — Constant gradient, outlier-robust
   * `huber` (Smooth L1) — Quadratic near 0, linear for big errors

> **Tip:** Set Runtime → Change runtime type → **T4 / L4 / A100 GPU**.

In [ ]:
# 1. Check GPU and install dependencies
!nvidia-smi
!pip install -q datasets pillow matplotlib
import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

### 2. Load the Workbench Scripts
Upload `data.py`, `models.py`, `losses.py`, `optimizers.py`, `train.py`, `inspect_data.py`, and `compare.py` into `/content/lab/` (or point `LAB_DIR` to your Google Drive folder).

In [ ]:
import os
from pathlib import Path

# Option A: Use local /content/lab (upload the 7 .py files there)
LAB_DIR = Path('/content/lab')
LAB_DIR.mkdir(parents=True, exist_ok=True)

# Option B: Uncomment to mount Google Drive instead
# from google.colab import drive
# drive.mount('/content/drive')
# LAB_DIR = Path('/content/drive/MyDrive/linuxGPT/01_micrograd')

os.chdir(LAB_DIR)
print('Working dir:', os.getcwd())
print('Files:', sorted(p.name for p in LAB_DIR.glob('*.py')))

---
## Part A — FairFace: Dataset Balance Audit & Classification Loss Comparison

In [ ]:
# Audit FairFace class balance across Origin (7), Gender (2), and Age Groups (9)
!python inspect_data.py --dataset fairface

from IPython.display import Image, display
display(Image('runs/fairface_balance.png'))

In [ ]:
# Run all 4 classification losses on FairFace with identical hyperparameters!
# Start with --max-samples 16000 (~1-2 min/run on L4/T4). Remove --max-samples for full 86k run.
COMMON = '--dataset fairface --model cnn --img-size 128 --activation relu --norm batchnorm --optimizer adamw --lr 1e-3 --batch-size 128 --epochs 8 --max-samples 16000 --seed 42'

!python train.py --run-name ff_loss_ce        --cls-loss cross_entropy   $COMMON
!python train.py --run-name ff_loss_smooth    --cls-loss label_smoothing $COMMON
!python train.py --run-name ff_loss_focal     --cls-loss focal           $COMMON
!python train.py --run-name ff_loss_mse1hot   --cls-loss mse_onehot      $COMMON

In [ ]:
# Compare all FairFace loss runs side-by-side + inspect per-origin accuracy
!python compare.py

import json
from pathlib import Path
from IPython.display import Image, display

display(Image('runs/comparison.png'))

print('\n=== Per-Origin Accuracy Breakdown ===')
for p in sorted(Path('runs').glob('ff_loss_*/metrics.json')):
    d = json.loads(p.read_text())
    print(f"\n{d['run_name']} (avg grad_norm={d['final']['grad_norm']}):")
    for grp, acc in d['final']['per_origin_acc'].items():
        print(f'  {grp:18s}: {acc:.3f}')

---
## Part B — UTKFace: Dataset Balance Audit & Regression Loss Comparison (`MSE` vs `MAE` vs `Huber`)

Point `UTK_DIR` to your extracted UTKFace folder containing `[age]_[gender]_[race]_[date].jpg` files.

In [ ]:
UTK_DIR = '/content/UTKFace'  # Update to your UTKFace images path

# 1. Audit UTKFace balance — compare its max/min imbalance ratio against FairFace!
!python inspect_data.py --dataset utkface --data-dir $UTK_DIR

from IPython.display import Image, display
display(Image('runs/utkface_balance.png'))

In [ ]:
# 2. Compare Regression Losses for continuous age: MSE vs MAE vs Huber
#    + Bonus: Huber + Focal loss to handle UTKFace's imbalanced origin classes!
UTK_COMMON = f'--dataset utkface --data-dir {UTK_DIR} --model cnn --img-size 128 --activation relu --norm batchnorm --optimizer adamw --lr 1e-3 --batch-size 128 --epochs 10 --seed 42'

!python train.py --run-name utk_reg_mse         --reg-loss mse   --cls-loss cross_entropy $UTK_COMMON
!python train.py --run-name utk_reg_mae         --reg-loss mae   --cls-loss cross_entropy $UTK_COMMON
!python train.py --run-name utk_reg_huber       --reg-loss huber --cls-loss cross_entropy $UTK_COMMON
!python train.py --run-name utk_reg_huber_focal --reg-loss huber --cls-loss focal         $UTK_COMMON

!python compare.py